# 01 · Crop-disease vision model → ExecuTorch (.pte) — v3 "real-world photos"
**Bharat AI-SoC 2026–27 · PS1 · IlaiNalam**

**Why v3:** v2 learned only from PlantVillage lab photos (one leaf, plain grey background), so on real photos it can confuse crops (healthy tomato → healthy grape). v3 keeps the **same 38 classes, same class order and same test split** (the app needs no code change; only the model file is replaced) and adds:
1. **Background swapping:** each lab leaf is cut out and pasted onto soil, grass, walls, hands, blurred field photos and textures, then rotated, resized, shaded, blurred and JPEG-compressed, so the model learns the leaf, not the grey background.
2. **Real field photos (PlantDoc):** about 2,300 real field photos of the same crops are mixed into training. The PlantDoc **test** photos are never trained on; they give the honest "real photo" accuracy for the report.
3. **"Is this a plant?" calibration:** measures how green real leaves are (lab + field) versus faces and textures, and the model's confidence on each, and writes the cut-offs to `plant_check.json` for the app.

**Kaggle setup (same as before):**
1. *Create ▸ New Notebook ▸ File ▸ Import Notebook* → upload this file (fresh notebook).
2. *Session options*: **Accelerator → GPU T4 x2**, **Internet → On** (needed to download PlantDoc and the test faces/textures).
3. *Add Input*: **"plantvillage dataset"** by *abdallahalidev*.
4. **Run All** — about 70–90 min.
5. Download **`ps1_vision_outputs_v3.zip`** from the Output panel.

*Optional:* to get a "v2 vs v3 on real photos" comparison for the report, upload your old `crop_disease_int8.pte` as a Kaggle dataset (*Add Input ▸ Upload*). The notebook finds it automatically. Skip this if short on time.

**App input contract (unchanged):** `float32 [1,3,224,224]`, RGB 0–1 → probabilities for 38 classes.

In [ ]:
# Uses Kaggle's pre-installed torch/torchvision. Do NOT pip-install torch or executorch in this kernel.
!pip install -q --default-timeout=120 --retries 10 kagglehub opencv-python-headless scikit-learn
import os, sys, json, random, glob, time, shutil, subprocess, re, io
import numpy as np, torch, torchvision, cv2
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator to GPU T4 x2")

In [ ]:
# ---- Config -------------------------------------------------------------
SEED, IMG_SIZE, BATCH, EPOCHS, LR = 42, 224, 64, 6, 1e-3
BG_SWAP_PROB   = 0.6     # share of lab photos that get a new background
PLANTDOC_REPEAT = 4      # PlantDoc train photos are few, so each is shown 4× per epoch
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK      = "/kaggle/working" if IS_KAGGLE else "/content"
OUT_DIR   = os.environ.get("OUT_DIR_OVERRIDE", f"{WORK}/out")
ENV_DIR   = os.environ.get("ET_ENV_DIR", "/tmp/et_env")
TMP       = os.environ.get("TMP_DIR", "/tmp/ps1")
FAST_TEST = os.environ.get("FAST_TEST") == "1"               # local smoke test only
if FAST_TEST: EPOCHS = 1
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(TMP, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

def sh(cmd, timeout=None, check=True):
    print("$", cmd)
    try: r = subprocess.run(cmd, shell=True, timeout=timeout).returncode
    except subprocess.TimeoutExpired: print("  …timed out"); r = -1
    if check: assert r == 0, f"command failed ({r}): {cmd}"
    return r

## 1 · PlantVillage — same classes, same split as v2

In [ ]:
if FAST_TEST:
    DATA_DIR = os.environ["FAST_DATA_DIR"]
else:
    cands = [p for p in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(p)]
    if not cands:
        import kagglehub
        root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
        cands = [p for p in glob.glob(os.path.join(root, "**", "color"), recursive=True) if os.path.isdir(p)]
    assert cands, "PlantVillage 'color' folder not found — add the dataset via Add Input"
    DATA_DIR = cands[0]
print("DATA_DIR:", DATA_DIR)

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
cls_idx = {c: i for i, c in enumerate(classes)}
def parse_label(raw):
    crop, _, cond = raw.partition("___")
    clean = lambda s: s.replace("_", " ").replace(",", "").strip()
    return {"raw": raw, "crop": clean(crop), "condition": clean(cond) or "unknown", "healthy": cond.lower() == "healthy"}
labels = [parse_label(c) for c in classes]
json.dump(labels, open(f"{OUT_DIR}/labels.json", "w"), indent=1)
crop_of = [l["crop"] for l in labels]
print(len(classes), "classes")

from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, Dataset
pv_all = ImageFolder(DATA_DIR)
assert pv_all.classes == classes
idx = list(range(len(pv_all))); random.Random(SEED).shuffle(idx)          # identical to v2
n_val = n_test = int(0.1 * len(idx))
val_idx, test_idx, train_idx = idx[:n_val], idx[n_val:n_val + n_test], idx[n_val + n_test:]
json.dump({"data_dir": DATA_DIR, "classes": classes, "train": train_idx, "val": val_idx, "test": test_idx},
          open(f"{OUT_DIR}/splits.json", "w"))
print(f"PlantVillage: train {len(train_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 2 · PlantDoc — real field photos of the same crops
2,598 photos from the web/fields, 27 classes; 25 of them match a PlantVillage class. Labels are a bit noisy (it is a real-world dataset). We train on its **train** folder and report accuracy on its **test** folder only.

In [ ]:
PD2PV = {
 "apple scab leaf": "Apple___Apple_scab", "apple leaf": "Apple___healthy", "apple rust leaf": "Apple___Cedar_apple_rust",
 "bell pepper leaf": "Pepper,_bell___healthy", "bell pepper leaf spot": "Pepper,_bell___Bacterial_spot",
 "blueberry leaf": "Blueberry___healthy", "cherry leaf": "Cherry_(including_sour)___healthy",
 "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
 "corn rust leaf": "Corn_(maize)___Common_rust_", "peach leaf": "Peach___healthy",
 "potato leaf early blight": "Potato___Early_blight", "potato leaf late blight": "Potato___Late_blight",
 "raspberry leaf": "Raspberry___healthy", "soyabean leaf": "Soybean___healthy", "squash powdery mildew leaf": "Squash___Powdery_mildew",
 "strawberry leaf": "Strawberry___healthy", "tomato early blight leaf": "Tomato___Early_blight",
 "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot", "tomato leaf": "Tomato___healthy",
 "tomato leaf bacterial spot": "Tomato___Bacterial_spot", "tomato leaf late blight": "Tomato___Late_blight",
 "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus", "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
 "tomato mold leaf": "Tomato___Leaf_Mold", "grape leaf": "Grape___healthy", "grape leaf black rot": "Grape___Black_rot",
}
norm = lambda s: re.sub(r"[\s_]+", " ", s).strip().lower()

if FAST_TEST:
    PD_DIR = os.environ.get("FAST_PD_DIR", "")
else:
    PD_DIR = f"{TMP}/PlantDoc-Dataset"
    if not os.path.isdir(f"{PD_DIR}/test"):
        sh(f"git clone --depth 1 -q https://github.com/pratikkayal/PlantDoc-Dataset {PD_DIR}", timeout=900, check=False)

def ok_image(p):
    try:
        with Image.open(p) as im: im.draft("RGB", (256, 256)); im.convert("RGB").load()
        return True
    except Exception: return False

def pd_items(split):
    out, skipped = [], set()
    root = os.path.join(PD_DIR, split)
    if not os.path.isdir(root): return out
    for d in sorted(os.listdir(root)):
        tgt = PD2PV.get(norm(d))
        if tgt is None or tgt not in cls_idx: skipped.add(d); continue
        for f in sorted(os.listdir(os.path.join(root, d))):
            p = os.path.join(root, d, f)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) and ok_image(p): out.append((p, cls_idx[tgt]))
    if skipped: print(f"  {split}: no PlantVillage match, skipped →", sorted(skipped))
    return out

pd_train_all, pd_test = pd_items("train"), pd_items("test")
HAVE_PD = len(pd_train_all) > 0 and len(pd_test) > 0
random.Random(SEED).shuffle(pd_train_all)
n_pdv = max(1, len(pd_train_all) // 10) if HAVE_PD else 0
pd_val, pd_train = pd_train_all[:n_pdv], pd_train_all[n_pdv:]
print(f"PlantDoc: train {len(pd_train)} · val {len(pd_val)} · test {len(pd_test)}" if HAVE_PD
      else "⚠️ PlantDoc not available — training continues with background swapping only (no real-photo numbers)")
json.dump([[p, y] for p, y in pd_test], open(f"{OUT_DIR}/plantdoc_test.json", "w"))
json.dump([[p, y] for p, y in pd_train[:60]], open(f"{OUT_DIR}/plantdoc_calib.json", "w"))

## 3 · Non-plant photos for the "Is this a plant?" check (faces + textures)
Used **only to measure** cut-offs (never trained on). Downloads are optional: if one fails, the notebook carries on.

In [ ]:
NEG = []   # list of RGB uint8 arrays (224×224)
if not FAST_TEST:
    try:   # faces — the exact case that fooled the app
        from sklearn.datasets import fetch_lfw_people
        lfw = fetch_lfw_people(color=True, resize=1.0, min_faces_per_person=0, download_if_missing=True, data_home=f"{TMP}/lfw")
        for im in lfw.images[np.random.RandomState(0).choice(len(lfw.images), 300, replace=False)]:
            im = im if im.max() > 1.5 else im * 255
            NEG.append(cv2.resize(im.astype(np.uint8), (224, 224)))
        print("faces:", len(NEG))
    except Exception as e: print("faces download skipped:", str(e)[:120])
    try:   # textures (walls, fabric, wood, …)
        sh(f"{sys.executable} -c \"import torchvision; torchvision.datasets.DTD('{TMP}/dtd', split='test', download=True)\"", timeout=600, check=False)
        tex = sorted(glob.glob(f"{TMP}/dtd/**/images/**/*.jpg", recursive=True))
        for p in [tex[i] for i in np.random.RandomState(1).choice(len(tex), min(300, len(tex)), replace=False)] if tex else []:
            im = np.array(Image.open(p).convert("RGB")); s = min(im.shape[:2])
            NEG.append(cv2.resize(im[:s, :s], (224, 224)))
        print("faces + textures:", len(NEG))
    except Exception as e: print("textures skipped:", str(e)[:120])
TEXTURES = sorted(glob.glob(f"{TMP}/dtd/**/images/**/*.jpg", recursive=True))   # also used as backgrounds

## 4 · Leaf cut-out (same algorithm as the app's `LeafAnalysis.kt`) + background swapping

In [ ]:
SEV = dict(BG_DIST=18, LESION_DIST=22, BORDER=6, KEEP_PIECE=0.15, MILD=0.10, MODERATE=0.25)
GREEN = dict(H_MIN=30, H_MAX=90, S_MIN=35)     # OpenCV hue 0–180 — same "greenish" test as the app

def _lab(im):
    return cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32) * np.array([100/255, 1, 1], np.float32) - np.array([0, 128, 128], np.float32)

def _fill_holes(m):
    h, w = m.shape; pad = np.zeros((h + 2, w + 2), np.uint8); pad[1:-1, 1:-1] = m
    ff = pad.copy(); cv2.floodFill(ff, np.zeros((h + 4, w + 4), np.uint8), (0, 0), 1)
    holes = (ff == 0)[1:-1, 1:-1]
    return m.astype(bool) | holes

def leaf_mask(rgb, cfg=SEV):
    L = _lab(rgb); h, w = L.shape[:2]; b = cfg["BORDER"]
    ring = np.zeros((h, w), bool); ring[:b] = ring[-b:] = True; ring[:, :b] = ring[:, -b:] = True
    bg = np.median(L[ring], 0); d_bg = np.sqrt(((L - bg) ** 2).sum(-1))
    fg = (d_bg > cfg["BG_DIST"]).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(fg, 8)
    if n <= 1: return np.zeros((h, w), bool), d_bg, L
    big = st[1:, cv2.CC_STAT_AREA].max()
    keep = np.isin(lab, [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= cfg["KEEP_PIECE"] * big])
    leaf = _fill_holes(keep.astype(np.uint8))
    if leaf.mean() > 0.97: leaf = np.ones((h, w), bool)
    return leaf, d_bg, L

def severity(rgb224, cfg=SEV):
    leaf, d_bg, L = leaf_mask(rgb224, cfg); h, w = leaf.shape
    if not leaf.any(): return 0.0, "No leaf", leaf, leaf
    core = cv2.erode(leaf.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV); H, S, V = [hsv[..., i].astype(int) for i in range(3)]
    greenish = core & (H >= GREEN["H_MIN"]) & (H <= GREEN["H_MAX"]) & (S >= GREEN["S_MIN"])
    ref = np.median(L[greenish], 0) if greenish.sum() > 50 else np.median(L[core], 0)
    shadow = ((V < 110) & (S < 55)) | (V < 45)
    les = core & (np.sqrt(((L - ref) ** 2).sum(-1)) > cfg["LESION_DIST"]) & (d_bg >= cfg["BG_DIST"]) & ~shadow
    les = cv2.morphologyEx(les.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8)).astype(bool)
    frac = float(les.sum()) / max(int(core.sum()), 1)
    return frac, ("Mild" if frac < cfg["MILD"] else "Moderate" if frac < cfg["MODERATE"] else "Severe"), les, leaf

def green_fraction(rgb224):
    """Share of the found 'leaf' that is plant-green. Real leaves: high. Faces, walls, clothes: near 0."""
    leaf, _, _ = leaf_mask(rgb224)
    if leaf.sum() < 0.05 * leaf.size: return 0.0
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV)
    g = (hsv[..., 0] >= GREEN["H_MIN"]) & (hsv[..., 0] <= GREEN["H_MAX"]) & (hsv[..., 1] >= GREEN["S_MIN"])
    return float((g & leaf).sum()) / float(leaf.sum())

def app_square(path_or_img, size=224):
    """Exactly what the app does: centre square → 512 → 224."""
    im = path_or_img if isinstance(path_or_img, np.ndarray) else np.array(_open(path_or_img))
    s = min(im.shape[:2]); y0 = (im.shape[0] - s) // 2; x0 = (im.shape[1] - s) // 2
    return cv2.resize(cv2.resize(im[y0:y0 + s, x0:x0 + s], (512, 512), interpolation=cv2.INTER_AREA), (size, size), interpolation=cv2.INTER_AREA)

def _open(p, draft=512):
    im = Image.open(p); im.draft("RGB", (draft, draft)); return im.convert("RGB")

PALETTE = np.array([[120,85,60],[90,65,45],[150,120,90],[70,55,40],      # soils
                    [60,110,50],[40,80,35],[100,140,70],                  # grass / foliage
                    [128,128,128],[200,200,200],[240,240,235],[60,60,60], # greys, paper, dark
                    [225,180,150],[160,110,80],                           # hands / skin
                    [150,190,230],[170,130,90],[200,170,120]], np.float32) # sky, wood, sand

def random_background(rng, size=256):
    r = rng.random()
    if r < 0.35 and pd_train:            # blurred real field photo
        im = np.array(_open(pd_train[rng.integers(len(pd_train))][0]))
        im = cv2.resize(im, (size, size)); k = int(rng.choice([7, 11, 15, 21]))
        return cv2.GaussianBlur(im, (k, k), 0)
    if r < 0.55 and TEXTURES:            # wall / fabric / wood texture
        return cv2.resize(np.array(_open(TEXTURES[rng.integers(len(TEXTURES))])), (size, size))
    if r < 0.70:                          # another lab leaf, heavily blurred (leafy clutter)
        im = np.array(_open(pv_all.samples[train_idx[rng.integers(len(train_idx))]][0]))
        return cv2.GaussianBlur(cv2.resize(im, (size, size)), (31, 31), 0)
    c1, c2 = PALETTE[rng.integers(len(PALETTE))], PALETTE[rng.integers(len(PALETTE))]
    t = np.linspace(0, 1, size)[:, None, None] if rng.random() < 0.5 else np.zeros((size, 1, 1))
    bg = c1 * (1 - t) + c2 * t + np.zeros((size, size, 3))
    blotch = cv2.resize(rng.normal(0, 1, (8, 8)).astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC)[..., None]
    grain = rng.normal(0, 1, (size, size, 1))          # same on all channels → natural texture, not rainbow speckle
    bg = bg * (1 + blotch * rng.uniform(0.03, 0.15)) + grain * rng.uniform(1, 8)
    return np.clip(bg, 0, 255).astype(np.uint8)

def swap_background(img, rng, size=256):
    a = cv2.resize(np.array(img), (size, size))
    leaf, _, _ = leaf_mask(a)
    if leaf.mean() < 0.05 or leaf.mean() > 0.95: return img
    m = cv2.GaussianBlur(leaf.astype(np.float32), (5, 5), 0)
    s = rng.uniform(0.55, 1.0); n = int(size * s)
    ang = rng.uniform(-180, 180); M = cv2.getRotationMatrix2D((size / 2, size / 2), ang, 1.0)
    a = cv2.resize(cv2.warpAffine(a, M, (size, size), borderValue=0), (n, n))
    m = cv2.resize(cv2.warpAffine(m, M, (size, size), borderValue=0), (n, n))[..., None]
    bg = random_background(rng, size).astype(np.float32)
    y, x = rng.integers(0, size - n + 1), rng.integers(0, size - n + 1)
    bg[y:y + n, x:x + n] = bg[y:y + n, x:x + n] * (1 - m) + a * m
    if rng.random() < 0.3:                # cast a shadow across part of the photo
        yy, xx = np.mgrid[0:size, 0:size]; th = rng.uniform(0, np.pi)
        side = (np.cos(th) * (xx - size / 2) + np.sin(th) * (yy - size / 2)) > rng.uniform(-60, 60)
        bg[side] *= rng.uniform(0.45, 0.8)
    return Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8))

class RandomJPEG:
    def __init__(self, p=0.4): self.p = p
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(25, 85)); b.seek(0); return Image.open(b).convert("RGB")

photo_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.45, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
                      T.ColorJitter(0.4, 0.4, 0.35, 0.04), T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.3),
                      RandomJPEG(0.4), T.ToTensor()])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor()])

class TrainSet(Dataset):
    def __init__(self): self.items = [(pv_all.samples[i][0], pv_all.samples[i][1], "pv") for i in train_idx] + \
                                     [(p, y, "pd") for p, y in pd_train] * PLANTDOC_REPEAT
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, y, src = self.items[k]; rng = np.random.default_rng(random.getrandbits(32))
        im = _open(p)
        if src == "pv":
            im = im.resize((256, 256))
            if rng.random() < BG_SWAP_PROB: im = swap_background(im, rng)
        return photo_tf(im), y

class AppSet(Dataset):          # field photos, preprocessed exactly like the app
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, y = self.items[k]
        return torch.from_numpy(app_square(p)).permute(2, 0, 1).float() / 255, y

train_set = TrainSet()
workers = 0 if FAST_TEST else 4
train_dl = DataLoader(train_set, BATCH, shuffle=True, num_workers=workers, pin_memory=True, drop_last=True,
                      persistent_workers=workers > 0)
pv_eval = ImageFolder(DATA_DIR, transform=eval_tf)
val_dl   = DataLoader(Subset(pv_eval, val_idx),  BATCH, num_workers=workers)
test_dl  = DataLoader(Subset(pv_eval, test_idx), BATCH, num_workers=workers)
pdval_dl = DataLoader(AppSet(pd_val),  BATCH, num_workers=workers) if HAVE_PD else None
pdtest_dl= DataLoader(AppSet(pd_test), BATCH, num_workers=workers) if HAVE_PD else None
print("training images per epoch:", len(train_set))

# Preview for the report: what the model sees during training
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 6, figsize=(15, 8))
for a in ax.flat:
    x, y = train_set[random.randrange(len(train_set))]
    a.imshow(x.permute(1, 2, 0).numpy()); a.set_title(classes[y].replace("___", "\n")[:34], fontsize=7); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_examples.png", dpi=110); plt.close()

## 5 · Model (unchanged architecture) and training
The best epoch is chosen on **both** lab accuracy and real-photo accuracy (PlantDoc val), not lab accuracy alone.

In [ ]:
MODEL_SRC = """
import torch, torch.nn as nn, torchvision
class Classifier(nn.Module):
    # Normalisation + backbone + (softmax when export_mode) -> the .pte takes 0-1 RGB and returns probabilities.
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        w = torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        self.net = torchvision.models.mobilenet_v3_large(weights=w)
        self.net.classifier[3] = nn.Linear(self.net.classifier[3].in_features, n_classes)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.export_mode = False
    def forward(self, x):
        logits = self.net((x - self.mean) / self.std)
        return torch.softmax(logits, dim=1) if self.export_mode else logits
"""
open(f"{WORK}/vision_model.py", "w").write(MODEL_SRC)
sys.path.insert(0, WORK)
from vision_model import Classifier
model = Classifier(len(classes), pretrained=not FAST_TEST).to(device)

import torch.nn as nn
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl))
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def predict(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append(torch.softmax(model(x.to(device)).float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y)
acc = lambda P, Y: (P.argmax(1) == Y).float().mean().item()

best = -1.0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0; n = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0); n += x.size(0)
    lab_acc = acc(*predict(val_dl))
    field_acc = acc(*predict(pdval_dl)) if HAVE_PD else float("nan")
    score = (lab_acc + field_acc) / 2 if HAVE_PD else lab_acc
    print(f"epoch {ep+1}/{EPOCHS} · loss {tot/n:.4f} · lab val {lab_acc:.4f} · real-photo val {field_acc:.4f} · {time.time()-t0:.0f}s")
    if score >= best:
        best = score; torch.save(model.state_dict(), f"{OUT_DIR}/best.pt")
model.load_state_dict(torch.load(f"{OUT_DIR}/best.pt", map_location=device))

## 6 · Test results — lab photos AND real photos (for the report)
* **Any crop:** the model picks from all 38 classes (how the app works today).
* **Crop known:** the model picks only among the diseases of the crop the farmer selects (if we add a "Which crop?" step to the app).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
P, Y = predict(test_dl)
metrics = {"version": "v3", "test_accuracy_pytorch": acc(P, Y), "macro_f1_pytorch": f1_score(Y, P.argmax(1), average="macro"),
           "n_test": len(test_idx)}
cm = confusion_matrix(Y, P.argmax(1), labels=list(range(len(classes))))
plt.figure(figsize=(12, 10)); plt.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues")
plt.title("Confusion matrix — PlantVillage test (row-normalised)"); plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150); plt.close()
metrics["per_class"] = classification_report(Y, P.argmax(1), labels=list(range(len(classes))), target_names=classes,
                                             output_dict=True, zero_division=0)

crop_mask = torch.tensor([[crop_of[j] == crop_of[i] for j in range(len(classes))] for i in range(len(classes))])
def field_report(P, Y):
    crop_known = (P * crop_mask[Y]).argmax(1)
    top3 = (P.topk(3, 1).indices == Y[:, None]).any(1).float().mean().item()
    wrong_crop = np.mean([crop_of[p] != crop_of[y] for p, y in zip(P.argmax(1).tolist(), Y.tolist())])
    return {"n": len(Y), "accuracy_any_crop": acc(P, Y), "top3_any_crop": top3,
            "accuracy_crop_known": (crop_known == Y).float().mean().item(), "wrong_crop_rate": float(wrong_crop)}
print(f"LAB  (PlantVillage test, {len(Y)} photos): accuracy {metrics['test_accuracy_pytorch']:.4f} · macro-F1 {metrics['macro_f1_pytorch']:.4f}")
if HAVE_PD:
    Pf, Yf = predict(pdtest_dl)
    metrics["plantdoc_test_pytorch"] = field_report(Pf, Yf)
    r = metrics["plantdoc_test_pytorch"]
    print(f"REAL (PlantDoc test, {r['n']} photos): any crop {r['accuracy_any_crop']:.3f} · top-3 {r['top3_any_crop']:.3f} · "
          f"crop known {r['accuracy_crop_known']:.3f} · wrong crop {r['wrong_crop_rate']:.3f}")
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1, default=float)

## 7 · "Is this a plant?" cut-offs → `plant_check.json`
Two checks the app will run before naming a disease:
1. **Green share:** what fraction of the found "leaf" is plant-green. The cut-off is set so that 99 % of real lab leaves and ~98 % of real field leaves pass (even badly diseased ones).
2. **Confidence:** the model's top probability. Reported for lab photos, real photos and non-plants so the app's "Not sure" threshold can be chosen from data.

In [ ]:
rs = np.random.RandomState(0)
pv_sample = [pv_all.samples[i][0] for i in rs.choice(test_idx, min(2000, len(test_idx)), replace=False)]
g_pv = np.array([green_fraction(app_square(p)) for p in pv_sample])
g_pd = np.array([green_fraction(app_square(p)) for p, _ in pd_test]) if HAVE_PD else np.array([])
g_neg = np.array([green_fraction(im) for im in NEG])
q = lambda a, ps=(0.5, 1, 2, 5, 10, 50): {f"p{p}": float(np.percentile(a, p)) for p in ps} if len(a) else {}

thr = float(np.percentile(g_pv, 1))
if len(g_pd): thr = min(thr, float(np.percentile(g_pd, 2)))
thr = float(np.clip(np.floor(thr * 100) / 100, 0.02, 0.25))

@torch.no_grad()
def conf_of(arrays):
    model.eval(); out = []
    for i in range(0, len(arrays), 64):
        x = torch.from_numpy(np.stack(arrays[i:i + 64])).permute(0, 3, 1, 2).float().div(255).to(device)
        out.append(torch.softmax(model(x).float(), 1).max(1).values.cpu())
    return torch.cat(out).numpy() if out else np.array([])
c_pv = P.max(1).values.numpy()
c_pd_ok = Pf.max(1).values[(Pf.argmax(1) == Yf)].numpy() if HAVE_PD else np.array([])
c_pd_bad = Pf.max(1).values[(Pf.argmax(1) != Yf)].numpy() if HAVE_PD else np.array([])
c_neg = conf_of(NEG)

plant_check = {
    "green_h_min": GREEN["H_MIN"], "green_h_max": GREEN["H_MAX"], "green_s_min": GREEN["S_MIN"],
    "min_green_fraction": thr,
    "stats": {
        "green_fraction": {"lab_leaves": q(g_pv), "field_leaves": q(g_pd), "non_plants": q(g_neg, (50, 90, 95, 99))},
        "pass_rate_at_cutoff": {"lab_leaves": float((g_pv >= thr).mean()),
                                "field_leaves": float((g_pd >= thr).mean()) if len(g_pd) else None,
                                "non_plants_rejected": float((g_neg < thr).mean()) if len(g_neg) else None},
        "confidence": {"lab": q(c_pv), "field_correct": q(c_pd_ok), "field_wrong": q(c_pd_bad, (50, 90)),
                       "non_plants": q(c_neg, (50, 90))},
    },
}
json.dump(plant_check, open(f"{OUT_DIR}/plant_check.json", "w"), indent=1)
print(json.dumps({k: v for k, v in plant_check.items() if k != "stats"}, indent=1))
print(json.dumps(plant_check["stats"]["pass_rate_at_cutoff"], indent=1))

## 8 · Severity figure (unchanged algorithm; for the report)

In [ ]:
diseased = [i for i in test_idx if not labels[pv_all.targets[i]]["healthy"]][:4]
fig, ax = plt.subplots(3, max(len(diseased), 1), figsize=(4 * max(len(diseased), 1), 12), squeeze=False)
for k, i in enumerate(diseased):
    img = app_square(pv_all.samples[i][0])
    frac, lab_, les, leaf = severity(img)
    grey = img.copy(); grey[~leaf] = (0.35 * grey[~leaf] + 0.65 * 230).astype(np.uint8)
    red = img.copy(); red[les] = (0.3 * red[les] + np.array([0.7 * 255, 0, 0])).astype(np.uint8)
    ax[0, k].imshow(img); ax[0, k].set_title(classes[pv_all.targets[i]][:28], fontsize=9)
    ax[1, k].imshow(grey); ax[1, k].set_title("1. leaf found (background removed)", fontsize=9)
    ax[2, k].imshow(red); ax[2, k].set_title(f"2. spots: {lab_} ({frac:.0%})", fontsize=9)
    for a in ax[:, k]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/severity_examples.png", dpi=120); plt.close()
json.dump(SEV, open(f"{OUT_DIR}/severity_config.json", "w"), indent=1)

## 9 · Export to ExecuTorch in a separate environment (same as v2)

In [ ]:
PINS = "executorch==1.6.0 torch==2.15.0 torchvision==0.29.2 torchao==0.18.0"   # tested together
if not os.path.exists(f"{ENV_DIR}/bin/python"):
    sh(f"{sys.executable} -m pip install -q virtualenv")
    sh(f"{sys.executable} -m virtualenv -q {ENV_DIR}")
pip = f"{ENV_DIR}/bin/pip install -q --default-timeout=120 --retries 10"
if not FAST_TEST:
    if subprocess.run(f"{pip} --extra-index-url https://download.pytorch.org/whl/cpu {PINS}", shell=True).returncode != 0:
        print("CPU wheel index failed — retrying from the standard index")
        sh(f"{pip} {PINS}")
sh(f"{ENV_DIR}/bin/python -c \"import torch, executorch; print('export env: torch', torch.__version__)\"")

In [ ]:
EXPORT_SRC = r"""
import sys, os, json, time, argparse, glob
import numpy as np, torch
from PIL import Image
from torch.export import export
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.runtime import Runtime

ap = argparse.ArgumentParser()
ap.add_argument("--work"); ap.add_argument("--out"); ap.add_argument("--n_eval", type=int, default=1500); ap.add_argument("--old_pte", default="")
a = ap.parse_args()
sys.path.insert(0, a.work)
from vision_model import Classifier

def app_tensor(p):   # centre square -> 512 -> 224, like the app
    im = Image.open(p); im.draft("RGB", (512, 512)); im = im.convert("RGB")
    w, h = im.size; s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    im = im.resize((512, 512), Image.BOX).resize((224, 224), Image.BOX)
    return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float().div(255)[None]

sp = json.load(open(f"{a.out}/splits.json"))
ds = ImageFolder(sp["data_dir"], transform=T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()]))
assert ds.classes == sp["classes"], "class order mismatch"
pd_test = json.load(open(f"{a.out}/plantdoc_test.json")); pd_cal = json.load(open(f"{a.out}/plantdoc_calib.json"))

m = Classifier(len(sp["classes"]), pretrained=False)
m.load_state_dict(torch.load(f"{a.out}/best.pt", map_location="cpu"))
m.export_mode = True; m.eval()
example = (torch.rand(1, 3, 224, 224),)

def to_pte(model, path):
    prog = to_edge_transform_and_lower(export(model, example), partitioner=[XnnpackPartitioner()]).to_executorch()
    open(path, "wb").write(prog.buffer)
    print(f"saved {path} ({len(prog.buffer)/1e6:.1f} MB)", flush=True)

to_pte(m, f"{a.out}/crop_disease_fp32.pte")

g = export(m, example).module()
prepared = prepare_pt2e(g, XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True)))
with torch.no_grad():                                      # calibrate on lab AND real photos (batch-1 graph)
    for x, _ in DataLoader(Subset(ds, sp["train"][:250]), 1): prepared(x)
    for p, _ in pd_cal: prepared(app_tensor(p))
to_pte(convert_pt2e(prepared), f"{a.out}/crop_disease_int8.pte")

def run(path, items):
    meth = Runtime.get().load_program(path).load_method("forward"); ok, times = 0, []
    for x, y in items:
        t = time.perf_counter(); p = meth.execute([x])[0]; times.append(time.perf_counter() - t)
        ok += int(p.argmax(1).item() == y)
    return ok / max(len(times), 1), 1000 * float(np.median(times)) if times else None

lab_items = lambda: ((x, y.item()) for x, y in DataLoader(Subset(ds, sp["test"][:a.n_eval]), 1))
field_items = lambda: ((app_tensor(p), y) for p, y in pd_test)
res = {}
for name in ["fp32", "int8"]:
    path = f"{a.out}/crop_disease_{name}.pte"
    lab_acc, ms = run(path, lab_items())
    res[f"pte_{name}"] = {"accuracy": lab_acc, "n_images": min(a.n_eval, len(sp["test"])), "median_ms_notebook_cpu": ms,
                          "size_mb": os.path.getsize(path) / 1e6}
    if pd_test: res[f"pte_{name}"]["plantdoc_accuracy"] = run(path, field_items())[0]
    print(name, res[f"pte_{name}"], flush=True)
if a.old_pte and pd_test:
    res["v2_int8_for_comparison"] = {"lab_accuracy": run(a.old_pte, lab_items())[0], "plantdoc_accuracy": run(a.old_pte, field_items())[0]}
    print("v2 (old model):", res["v2_int8_for_comparison"], flush=True)
json.dump(res, open(f"{a.out}/pte_metrics.json", "w"), indent=1)
"""
open(f"{WORK}/export_vision.py", "w").write(EXPORT_SRC)
old = [p for p in glob.glob("/kaggle/input/**/crop_disease_int8.pte", recursive=True)]
print("old v2 model for comparison:", old[0] if old else "not attached (optional)")
sh(f"{ENV_DIR}/bin/python {WORK}/export_vision.py --work {WORK} --out {OUT_DIR} --n_eval {40 if FAST_TEST else 1500}"
   + (f" --old_pte '{old[0]}'" if old else ""))

metrics.update(json.load(open(f"{OUT_DIR}/pte_metrics.json")))
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1, default=float)
print(json.dumps({k: v for k, v in metrics.items() if k != "per_class"}, indent=1, default=float))

## 10 · Package the outputs

In [ ]:
keep = ["crop_disease_int8.pte", "crop_disease_fp32.pte", "labels.json", "severity_config.json", "plant_check.json",
        "metrics.json", "confusion_matrix.png", "severity_examples.png", "training_examples.png"]
pkg = f"{WORK}/ps1_vision_outputs_v3"
os.makedirs(pkg, exist_ok=True)
for f in keep: shutil.copy(f"{OUT_DIR}/{f}", pkg)
print("zipped →", shutil.make_archive(pkg, "zip", pkg))
i8 = metrics["pte_int8"]
print(f"\nINT8 model: {i8['size_mb']:.1f} MB · lab accuracy {i8['accuracy']:.4f}"
      + (f" · REAL-photo accuracy {i8['plantdoc_accuracy']:.4f}" if "plantdoc_accuracy" in i8 else ""))
if "v2_int8_for_comparison" in metrics: print("v2 for comparison:", metrics["v2_int8_for_comparison"])
print("plant check cut-off (min green share):", plant_check["min_green_fraction"])